# Урок 4. Задание 2: эмбеддинги и векторный поиск

Заполненный прекод из курса. Пропуски `# Ваш код здесь` дописаны.

## Пересобираем `chunks` из урока 3

In [1]:
def chunk_text(text: str, max_chars: int = 800, min_chars: int = 50) -> list[str]:
    """Режет Markdown по заголовкам ##; длинные секции дробит по параграфам.

    Параметры:
        text: исходный Markdown-текст
        max_chars: максимальная длина одного чанка в символах
        min_chars: минимальная длина чанка - чанки короче выбрасываются

    Возвращает:
        Список текстовых чанков
    """
    lines = text.split("\n")
    sections, current = [], []
    for line in lines:
        if line.startswith("## ") and current:
            sections.append("\n".join(current).strip())
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append("\n".join(current).strip())

    chunks = []
    for section in sections:
        if not section:
            continue
        if len(section) <= max_chars:
            chunks.append(section)
            continue
        # Длинную секцию дробим по двойным переносам (параграфам)
        buf = ""
        for paragraph in section.split("\n\n"):
            if len(buf) + len(paragraph) + 2 <= max_chars:
                buf = f"{buf}\n\n{paragraph}" if buf else paragraph
            else:
                if buf:
                    chunks.append(buf.strip())
                buf = paragraph
        if buf:
            chunks.append(buf.strip())

    return [c for c in chunks if len(c) >= min_chars]

In [2]:
from pathlib import Path

DOCS_DIR = Path("docs")

chunks = []
for path in sorted(DOCS_DIR.rglob("*")):
    if path.suffix.lower() in {".md", ".mdx"}:
        text = path.read_text(encoding="utf-8")
        for chunk in chunk_text(text):
            chunks.append({"text": chunk, "source": str(path)})

print(f"Всего чанков: {len(chunks)}")

Всего чанков: 530


## Решение задания

In [3]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 1. Загружаем модель эмбеддингов
embed_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 2. Считаем эмбеддинги для всех чанков
texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)

# 3. Строим FAISS-индекс
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

# 4. Функция поиска
def vector_search(question: str, top_k: int = 3) -> list[dict]:
    query_emb = embed_model.encode(
        [question],
        normalize_embeddings=True,
    ).astype(np.float32)
    scores, indices = index.search(query_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "text": chunks[idx]["text"],
            "source": chunks[idx]["source"],
            "score": float(score),
        })
    return results

# 5. Проверяем на трёх вопросах
questions = [
    "What HTTP method does /api/generate use?",
    "Как сменить папку, где хранятся модели?",
    "What is a Modelfile?",
]

for q in questions:
    print(f"=== {q} ===")
    for r in vector_search(q, top_k=3):
        print(f"  [{r['source']}]  score={r['score']:.3f}")
        print(f"  {r['text'][:120].strip()}...\n")


/Users/doskhanstybayev/Desktop/Projects/ai_assistant_free_track/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7119.84it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Batches:   6%|▌         | 1/17 [00:01<00:16,  1.03s/it]

...

Batches: 100%|██████████| 17/17 [00:04<00:00,  6.04it/s]

Batches: 100%|██████████| 17/17 [00:04<00:00,  4.20it/s]

=== What HTTP method does /api/generate use? ===


  [docs/api/introduction.mdx]  score=0.466
  ## Base URL

After installation, Ollama's API is served by default at:

```
http://localhost:11434/api
```

For running...

  [docs/faq.mdx]  score=0.461
  ```shell
docker build -t ollama-with-ca .
docker run -d -e HTTPS_PROXY=https://my.proxy.example.com -p 11434:11434 ollam...

  [docs/api/introduction.mdx]  score=0.443
  ## Example request

Once Ollama is running, its API is automatically available and can be accessed via `curl`:

```shell...

=== Как сменить папку, где хранятся модели? ===
  [docs/faq.mdx]  score=0.586
  ## Where are models stored?

- macOS: `~/.ollama/models`
- Linux: `/usr/share/ollama/.ollama/models`
- Windows: `C:\User...

  [docs/windows.mdx]  score=0.581
  To change where Ollama stores the downloaded models instead of using your home directory, set the environment variable `...

  [docs/modelfile.mdx]  score=0.440
  ```shell
ollama show --modelfile llama3.2
```

```
# Modelfile generated by "ollama show"
# To build

  [docs/modelfile.mdx]  score=0.762
  ---
title: Modelfile Reference
---

A Modelfile is the blueprint to create and share customized models using Ollama....

  [docs/import.mdx]  score=0.628
  - a model from Ollama
- a GGUF file
- a Safetensors based model

Once you have created your `Modelfile`, use the `ollama...

  [docs/api.md]  score=0.571
  ```json5
{
  modelfile: '# Modelfile generated by "ollama show"\n# To build a new Modelfile based on this one, replace t...

